In [0]:
# Bibliotecas básicas
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Estatística
from scipy import stats

## Utilidades
%pip install openpyxl

## 1 - Carregamento e visualização inicial do dataset 
[Telco customer churn: IBM dataset](https://www.kaggle.com/datasets/yeanzc/telco-customer-churn-ibm-dataset)

In [0]:
import pandas as pd

df = pd.read_excel("../data/Telco_customer_churn.xlsx")

df.head()
df.info()
df.head(5)

## 2 - Análise e tratamento inicial
**Objetivo**: Padronizar nome das colunas e revisão inicial dos dados

In [0]:
# 1. Ajuste de nomes para facilitar a leitura e manipulação
df.columns = [c.lower().replace(" ", "_") for c in df.columns]

# 2. Verificar se há valores ausentes
print("\n===> Análise de valores ausentes: ")
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0]
missing_values = missing_values.sort_values(ascending=False)

if len(missing_values) > 0:
    print(missing_values)
else:
    print("Não há valores ausentes.")

# 3. Verificar se há valores duplicados
print("\n===> Análise de valores duplicados: ")
duplicated_rows = df.duplicated().sum()
if duplicated_rows > 0:
    print(f"\nTotal de linhas duplicadas: {duplicated_rows}")
else:
    print("Não há valores duplicados.")


# 4. Verificar se há colunas de texto que só têm espaços ou estão vazias
print("\n===> Análise de valores vazios: ")
for col in df.select_dtypes(include=['object']).columns:
    empty_strings = df[df[col].astype(str).str.strip() == ""].shape[0]
    if empty_strings > 0:
        print(f"A coluna '{col}' possui {empty_strings} valores vazios (strings de espaço).")


### Análise das integridade encontradas

In [0]:
# 4.1 Filtra apenas onde o total_charges está vazio
anomalias = df[df['total_charges'].astype(str).str.strip() == ""]
display(anomalias)

**Entendendo a correlação dos dados**

In [0]:
# 1. Flag temporária para identificar o vazio (espaço em branco)
df['is_total_charges_empty'] = df['total_charges'].astype(str).str.strip() == ""

# 2. Cruzamento da flag com a coluna tenure_months
analise_causa_raiz = df.groupby('is_total_charges_empty')['tenure_months'].agg(
    qtd_clientes='count',
    tenure_minimo='min',
    tenure_maximo='max',
    tenure_medio='mean'
).reset_index()

# Exibindo o resultado
print("Análise de Correlação: Total Charges Vazio vs. Tenure")
display(analise_causa_raiz)

# 3. Verificação visual direta dos 11 casos
colunas_validacao = ['customerid', 'tenure_months', 'monthly_charges', 'total_charges', 'contract']
display(df[df['is_total_charges_empty'] == True][colunas_validacao])

### Análise de integridade
Durante a análise de integridade, foram identificados **11 registros** na coluna `total_charges` contendo strings vazias (`" "`).

**Causa Raiz**:
Ao cruzar esses dados, foi observado que todos esses 11 registros pertencem a clientes com **tenure_months = 0**. Isso indica clientes recém-adquiridos que ainda não possuem um histórico de faturamento acumulado.

**Ação necessária**:
- Converter valores vazios para **0.0**, garantindo a consistência numérica da coluna.

In [0]:
# Correção das anomalias
df['total_charges'] = pd.to_numeric(df['total_charges'].replace(" ", 0))
#df = df.drop(columns=['is_total_charges_empty'])
display(df['total_charges'])

## Análise Exploratória de Dados e Estratégia de Modelagem
**Objetivo:** Analisar os fatores que influenciam o cancelamento de clientes (Churn) e preparar o dataset para a prototipagem.

### Insights Iniciais:
- O dataset apresenta um desbalanceamento de classes (~26% Churn).
- Variável Target: `Churn` (Sim/Não).

## 1 - Análise de Missing Values
**Objetivo**: Identificar valores ausentes e entender padrões de distribuição

In [0]:
# Análise de valores ausentes
print("=== ANÁLISE DE MISSING VALUES ===\n")

missing_values = pd.DataFrame({
    'column': df.columns,
    'missing_count': df.isnull().sum(),
    'missing_percentage': (df.isnull().sum() / len(df) * 100).round(2)
})

missing_values = missing_values[missing_values['missing_count'] > 0].sort_values(
    by='missing_percentage', ascending=False
)

if len(missing_values) > 0:
    print(missing_values)
    
    plt.figure(figsize=(10, 6))
    plt.barh(missing_values['column'], missing_values['missing_percentage'], color='coral')
    plt.xlabel('Porcentagem de Missing Values (%)')
    plt.title('Distribuição de Missing Values por Coluna')
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum missing value detectado!")

## 3 - Análise da Variável Target (Churn)
**Objetivo**: Entender a distribuição da variável alvo (balanceamento de classes)

In [0]:
df.rename(columns={'churn_value': 'target'}, inplace=True)

# Converter target para binário (0 = non-churn, 1 = churn)
# No dataset original, valores > 0 indicam presença de doença
df['target'] = (df['target'] > 0).astype(int)

print("=== DISTRIBUIÇÃO DA VARIÁVEL TARGET ===\n")
target_counts = df['target'].value_counts()
target_percentages = df['target'].value_counts(normalize=True) * 100

print("Contagem:")
print(target_counts)
print("\nPercentual:")
for idx, pct in target_percentages.items():
    label = "non-churn" if idx == 0 else "churn"
    print(f"{label} ({idx}): {pct:.2f}%")

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
axes[0].bar(['non-churn', 'churn'], target_counts.values, color=['lightblue', 'coral'])
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição da Variável Target')
axes[0].grid(axis='y', alpha=0.3)

# Gráfico de pizza
axes[1].pie(target_counts.values, labels=['non-churn', 'churn'], 
            autopct='%1.1f%%', colors=['lightblue', 'coral'], startangle=90)
axes[1].set_title('Proporção de Classes')

plt.tight_layout()
plt.show()

# Verificar se há desbalanceamento
ratio = target_counts.min() / target_counts.max()
print(f"\nRatio de balanceamento: {ratio:.2f}")
if ratio < 0.5:
    print("⚠️ Dataset desbalanceado! Considere usar técnicas como SMOTE ou class_weight.")
else:
    print("✓ Dataset razoavelmente balanceado.")

**⚠️ Observação sobre Desbalanceamento:**

Foi identificado que a classe minoritária **(Churn = Sim)** representa apenas 26,5% dos dados.
Ou seja, um desbalanceamento de aproximadamente 3:1 (73,5% vs 26,5%). 
Para o contexto de Churn, isso é considerado um desbalanceamento **moderado** mas exige cuidados para que o modelo não fique viciado na classe majoritári.
 
**Impacto:**
1. A métrica principal não poderá ser a **Acurácia**, considerar **F1-Score** ou **AUC-ROC**.
2. No treinamento, utilizar **Stratified Sampling** para garantir a proporção representativa das classes tanto no treino quanto no teste.
    ```
    from sklearn.model_selection import train_test_split

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )
    ```
3. Avaliaremos o uso de `class_weight='balanced'` nos modelos de baseline.
4. Observar a matriz de confusão para entender os erros.

**Técnicas de Rebalanceamento sugerida**
- **Class Weight (Peso de Classe)**: Em vez de mexer nos dados, dizer ao algoritmo que errar a classe `churn` custa mais caro.


## 4 - Análise de Outliers
**Objetivo**: Identificar valores extremos que podem ser erros de medição ou casos especiais

In [0]:
# Colunas numéricas (excluindo id e target)
numeric_cols = df.select_dtypes(include=[np.number]).columns
numeric_cols = numeric_cols.drop(['CustomerID', 'target'], errors='ignore')

n = len(numeric_cols)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4 * nrows))
axes = axes.ravel()

for idx, col in enumerate(numeric_cols):
    sns.boxplot(x=df[col], ax=axes[idx], color='coral')
    axes[idx].set_title(f'Boxplot: {col}', fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].grid(axis='x', alpha=0.3)
    col_zscore = np.abs(stats.zscore(df[col].dropna()))
    outlier_count = (col_zscore > 3).sum()
    axes[idx].text(0.95, 0.95, f'Outliers: {outlier_count}', 
                   transform=axes[idx].transAxes, fontsize=9,
                   verticalalignment='top', horizontalalignment='right',
                   bbox=dict(facecolor='white', alpha=0.5, edgecolor='gray'))

for ax in axes[n:]:
    fig.delaxes(ax)

plt.tight_layout()
plt.show()

### Conclusão da análise de Outliers

Para esta análise, utilizamos duas abordagens complementares:
1. **Visual (Boxplots):** Para observar a dispersão dos dados e a presença de valores extremos além dos quartis.
2. **Estatística (Z-Score):** Calculamos a quantidade de registros que se afastam mais de 3 desvios padrão da média.

### Conclusões:
- **Consistência:** Conforme indicado nos gráficos, o contador de outliers (Z-Score > 3) resultou em **0** para todas as variáveis numéricas.
- **Distribuição:** As variáveis apresentam distribuições bem delimitadas, sem a necessidade de técnicas de truncamento (clipping) ou remoção de registros por valores discrepantes.
- **Variável 'count':** Confirmada como uma constante unitária, sem variabilidade estatística.

> **Sugestão:** Mantere todos os dados originais, dado que não há ruído ou erros de entrada de dados detectados nesta fase.

## 5 - Análise de Anomalias e Valores Inválidos
**Objetivo**: Identificar valores que não fazem sentido para o domínio escolhido

In [0]:
anomalies = []

# Variáveis categóricas com valores fora do esperado
categorical_checks = {
    'gender': ['Male', 'Female'],
    'senior_citizen': ['Yes', 'No'],
    'partner': ['Yes', 'No'],
    'dependents': ['Yes', 'No'],
    'phone_service': ['Yes', 'No'],
    'multiple_lines': ['Yes', 'No', 'No phone service'],
    'internet_service': ['DSL', 'Fiber optic', 'No'],
    'online_security': ['Yes', 'No', 'No internet service'],
    'online_backup': ['Yes', 'No', 'No internet service'],
    'device_protection': ['Yes', 'No', 'No internet service'],
    'tech_support': ['Yes', 'No', 'No internet service'],
    'streaming_tv': ['Yes', 'No', 'No internet service'],
    'streaming_movies': ['Yes', 'No', 'No internet service'],
    'contract': ['Month-to-month', 'One year', 'Two year'],
    'paperless_billing': ['Yes', 'No'],
    'payment_method': ['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)'],
    'churn_label': ['Yes', 'No']
}

for col, valid_values in categorical_checks.items():
    if col in df.columns:
        invalid = df[~df[col].isin(valid_values) & df[col].notna()]
        if len(invalid) > 0:
            anomalies.append(f"{col}: {len(invalid)} valores inválidos")
            print(f"⚠️ {col}: {len(invalid)} valores fora do domínio esperado {valid_values}")

if len(anomalies) == 0:
    print("✓ Nenhuma anomalia óbvia detectada nas validações de domínio!")
else:
    print(f"\n📊 Total de tipos de anomalias detectadas: {len(anomalies)}")